# Bayesian Inference Demo (Python)

This notebook demonstrates Bayesian modeling capabilities using the `datascienceutils` library with Python bindings.

## Features Demonstrated:
- MCMC Sampling (Metropolis-Hastings)
- Convergence Diagnostics (R-hat, ESS)
- Bayesian A/B Testing
- Visualization with Matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import datascienceutils as dsu

# Configure plotting
plt.style.use('seaborn-v0_8')
np.random.seed(42)

## 1. MCMC Sampling - Normal Distribution

Sample from a standard normal distribution using Metropolis-Hastings implemented in Rust.

In [ ]:
# Define log posterior for N(0, 1)
def log_posterior(x):
    # Log of exp(-x^2/2)
    return -0.5 * x[0] * x[0]

initial = np.array([0.0])
proposal_std = 1.0
num_samples = 5000

print("Running MCMC sampler...")
samples = dsu.mcmc_sample(log_posterior, initial, proposal_std, num_samples)

print("=== MCMC Sampling Results ===")
print(f"Number of samples: {samples.shape[0]}")
print(f"Sample mean: {np.mean(samples):.4f} (expected: 0.0)")
print(f"Sample std: {np.std(samples):.4f} (expected: 1.0)")

In [ ]:
# Visualize the posterior
plt.figure(figsize=(10, 6))
plt.hist(samples.flatten(), bins=50, density=True, alpha=0.7, label='MCMC Samples')

x = np.linspace(-4, 4, 100)
y = 1/np.sqrt(2*np.pi) * np.exp(-0.5 * x**2)
plt.plot(x, y, 'r-', lw=2, label='True Distribution')

plt.title('Posterior Distribution (Standard Normal)')
plt.xlabel('Value')
plt.ylabel('Density')
plt.legend()
plt.show()

## 2. Convergence Diagnostics

Check MCMC convergence using R-hat and Effective Sample Size.

In [ ]:
rhat = dsu.compute_rhat(samples)
ess = dsu.effective_sample_size(samples)

print("\n=== Convergence Diagnostics ===")
print(f"R-hat: {rhat:.4f} (should be < 1.1)")
print(f"Effective Sample Size: {ess:.0f}")
print(f"ESS ratio: {(ess / len(samples)) * 100:.2f}%")

if rhat < 1.1:
    print("✓ Chain has converged!")
else:
    print("⚠ Chain may not have converged")

## 3. Bayesian A/B Testing

Compare conversion rates between two groups using Bayesian inference.

In [ ]:
# Generate synthetic data
# Control: 10% conversion rate
control = np.zeros(1000)
control[:100] = 1.0
np.random.shuffle(control)

# Treatment: 12% conversion rate
treatment = np.zeros(1000)
treatment[:120] = 1.0
np.random.shuffle(treatment)

print("=== A/B Test Setup ===")
print(f"Control: {len(control)} visitors, {np.sum(control):.0f} conversions ({np.mean(control)*100:.1f}%)")
print(f"Treatment: {len(treatment)} visitors, {np.sum(treatment):.0f} conversions ({np.mean(treatment)*100:.1f}%)")

In [ ]:
# Run Bayesian A/B Test
result = dsu.bayesian_ab_test(control, treatment)

print("\n=== Bayesian A/B Test Results ===")
print(f"Probability treatment is better: {result.prob_treatment_better * 100:.1f}%")
print(f"Expected lift: {result.expected_lift * 100:.2f}% points")
print(f"95% Credible Interval: [{result.credible_interval[0] * 100:.2f}%, {result.credible_interval[1] * 100:.2f}%]")

if result.prob_treatment_better > 0.95:
    print("\n✓ Strong evidence for treatment!")
elif result.prob_treatment_better > 0.80:
    print("\n⚠ Moderate evidence for treatment.")
else:
    print("\n✗ Insufficient evidence.")